In [1]:
import os
import warnings
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False

import data_utils
from base_utils_qwen import competition_scorer, evaluate_holdout, plot_training_curves
from baselines_utils import (
    make_baseline_pipeline,
    build_feature_extractor,
    build_classifier,
    RidgeRocketClassifier,
    HAS_TF,
)

In [2]:
# ============================================================
# CONFIGURATION — switch feature + classifier pipelines here
# ============================================================
TRAIN_DUMMY = False
TRAIN_RF = True
TRAIN_ROCKET = False
TRAIN_CNN = False  # requires tensorflow

# Feature extraction mode per model family:
#   tabular_simple | tabular_honeycomb | temporal_honeycomb | temporal_raw
FEATURE_MODE_TABULAR = "tabular_honeycomb"
FEATURE_MODE_ROCKET = "temporal_honeycomb"
FEATURE_MODE_CNN = "temporal_honeycomb"

# Tabular augmentation (RF only — applied inside pipeline when True)
use_tabular_augment = False
augment_kwargs = {"use_gaussian_noise": True, "noise_std": 0.01}
baseline_kwargs = {"strategy": "stratified"}
feature_kwargs = {}

target_col = "bfrb"
search_mode = "grid"  # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 10
train_size = 0.2  # lower for quick local tests; raise for full runs
error_score_constant = 0.0
verbose = 3
do_cross_val = False

# Use eg.csv sample for fast smoke tests (set False for full train.csv)
use_eg_sample = False
eg_sample_pct = 0.02

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

In [3]:
# ============================================================
# DATA LOADING & PREPROCESSING
# ============================================================
data_root = data_utils.find_data_root()
eg_path = data_root / "eg.csv"

if use_eg_sample and eg_path.exists():
    raw_train_df = pd.read_csv(eg_path)
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    print(f"Using eg.csv sample: {raw_train_df['sequence_id'].nunique()} sequences")
else:
    raw_train_df = pd.read_csv(data_root / "train.csv")
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    if use_eg_sample:
        seq_ids = raw_train_df["sequence_id"].drop_duplicates().sample(frac=eg_sample_pct, random_state=random_state)
        raw_train_df = raw_train_df[raw_train_df["sequence_id"].isin(seq_ids)].copy()
        print(f"Using {eg_sample_pct:.0%} sequence sample: {raw_train_df['sequence_id'].nunique()} sequences")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness correction
train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
left_handed_mask = train_df["handedness"].eq(0)
train_df.loc[left_handed_mask, "acc_x"] *= -1.0

# Upside-down correction
upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0
train_df = train_df.drop(columns=["handedness"])

# Targets
train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")

# Split
train_sample_df, hold_out_df = data_utils.sample_balanced_split(
    train_df, train_pct=train_size, test_pct=min(0.18, 1 - train_size), random_state=random_state
)

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = train_sample_df[["sequence_id", "is_target", "bfrb"]].copy()
y_test = hold_out_df[["sequence_id", "is_target", "bfrb"]].copy()
groups = X_train["sequence_id"]

Using local data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data
Train: 1458 seqs | 17.9%
Test:  1452 seqs  | 17.8%


In [4]:
# ============================================================================
# PARAMETER SPACE DEFINITION — INCLUDES FEATURE EXTRACTION
# ============================================================================

if search_mode == "bayesian":
    
    # ===== Random Forest (Tabular Honeycomb) =====
    rf_param_space = {
        # ---- Feature extraction (Honeycomb) ----
        "extractor__honeycomb_kwargs__acc_modes": Categorical([
            "raw", "raw|velocity", "smoothed|velocity|jerk"
        ]),
        "extractor__honeycomb_kwargs__rotation_modes": Categorical([
            "quaternion", "quaternion|angular_velocity", "quaternion|euler"
        ]),
        "extractor__honeycomb_kwargs__tof_modes": Categorical([
            "sensor_stats", "pooled_stats", "sensor_stats|pooled_diff"
        ]),
        "extractor__honeycomb_kwargs__thm_modes": Categorical([
            "centered_diff", "diff", "centered"
        ]),
        "extractor__honeycomb_kwargs__sampling_rate": Integer(10, 200),
        "extractor__honeycomb_kwargs__window_size": Integer(5, 50),
        "extractor__honeycomb_kwargs__clip_value": Categorical([None, 50.0, 100.0]),
        "extractor__honeycomb_kwargs__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__honeycomb_kwargs__motion_filter_mode": Categorical([None, "kalman", "extended_kalman"]),
        "extractor__honeycomb_kwargs__use_dead_reckoning": Categorical([True, False]),
        # ---- Random Forest classifier ----
        "classifier__base_estimator__n_estimators": Integer(10, 500),
        "classifier__base_estimator__max_depth": Integer(5, 25),
        "classifier__base_estimator__min_samples_leaf": Integer(1, 10),
        "classifier__base_estimator__class_weight": Categorical(["balanced", None]),
    }
    
    # ===== MiniRocket (Temporal Honeycomb) =====
    rocket_param_space = {
        # ---- Feature extraction (SequenceTensorExtractor) ----
        "extractor__acc_modes": Categorical([
            "raw", "raw|velocity", "smoothed|velocity|jerk"
        ]),
        "extractor__rotation_modes": Categorical([
            "quaternion", "quaternion|angular_velocity", "quaternion|euler"
        ]),
        "extractor__tof_modes": Categorical([
            "sensor_stats", "pooled_stats", "sensor_stats|pooled_diff"
        ]),
        "extractor__thm_modes": Categorical([
            "centered_diff", "diff", "centered"
        ]),
        "extractor__sampling_rate": Integer(10, 200),
        "extractor__maxlen": Integer(30, 200),
        "extractor__window_size": Integer(5, 50),
        "extractor__clip_value": Categorical([None, 50.0, 100.0]),
        "extractor__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__motion_filter_mode": Categorical([None, "kalman", "extended_kalman"]),
        "extractor__use_dead_reckoning": Categorical([True, False]),
        "extractor__dead_reckoning_detrend": Categorical([True, False]),
        "extractor__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        "extractor__kalman_measurement_noise": Real(1e-2, 5e-1, prior="log-uniform"),
        # ---- MiniRocket classifier ----
        "classifier__base_estimator__num_kernels": Integer(84, 5000),
        "classifier__base_estimator__alpha": Real(1e-4, 1e2, prior="log-uniform"),
        "classifier__base_estimator__feature_selection_percentile": Categorical([None, 25, 50]),
        "classifier__base_estimator__class_weight": Categorical(["balanced", None]),
    }
    
    # ===== 1D CNN (Temporal Honeycomb) =====
    cnn_param_space = {
        # ---- Feature extraction (SequenceTensorExtractor) ----
        "extractor__acc_modes": Categorical([
            "raw", "raw|velocity", "smoothed|velocity|jerk"
        ]),
        "extractor__rotation_modes": Categorical([
            "quaternion", "quaternion|angular_velocity", "quaternion|euler"
        ]),
        "extractor__tof_modes": Categorical([
            "sensor_stats", "pooled_stats", "sensor_stats|pooled_diff"
        ]),
        "extractor__thm_modes": Categorical([
            "centered_diff", "diff", "centered"
        ]),
        "extractor__sampling_rate": Integer(10, 200),
        "extractor__maxlen": Integer(60, 200),
        "extractor__window_size": Integer(5, 50 ),
        "extractor__clip_value": Categorical([None, 50.0, 100.0]),
        "extractor__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__motion_filter_mode": Categorical([None, "kalman", "extended_kalman"]),
        "extractor__use_dead_reckoning": Categorical([True, False]),
        "extractor__dead_reckoning_detrend": Categorical([True, False]),
        "extractor__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        "extractor__kalman_measurement_noise": Real(1e-2, 5e-1, prior="log-uniform"),
        # ---- CNN classifier ----
        "classifier__base_estimator__filters": Categorical(["32-64", "64-128", "128-256"]),
        "classifier__base_estimator__kernels": Categorical(["3-3", "5-3", "7-5-3"]),
        "classifier__base_estimator__pools": Categorical(["none", "2", "2-2"]),
        "classifier__base_estimator__dropout": Real(0.1, 0.5),
        "classifier__base_estimator__spatial_dropout": Real(0.0, 0.3),
        "classifier__base_estimator__l2_reg": Real(1e-5, 1e-2, prior="log-uniform"),
        "classifier__base_estimator__learning_rate": Real(1e-4, 1e-2, prior="log-uniform"),
        "classifier__base_estimator__batch_size": Categorical([16, 32, 64]),
        "classifier__base_estimator__epochs": Categorical([100]),
        "classifier__base_estimator__patience": Categorical([10]),
    }

else:  # GRID SEARCH
    
    # ===== Random Forest (Tabular Honeycomb) =====
    rf_param_space = {
        "extractor__honeycomb_kwargs__acc_modes": ["smoothed|velocity|jerk", 'raw'],
        "extractor__honeycomb_kwargs__rotation_modes": ["quaternion|angular_velocity"],
        "extractor__honeycomb_kwargs__tof_modes": ["pooled_stats"],
        "extractor__honeycomb_kwargs__thm_modes": ["centered_diff"],
        "extractor__honeycomb_kwargs__sampling_rate": [20, 100],
        "extractor__honeycomb_kwargs__window_size": [40],
        "extractor__honeycomb_kwargs__clip_value": [50.0],
        "extractor__honeycomb_kwargs__interp_mode": ["linear"],
        "extractor__honeycomb_kwargs__motion_filter_mode": ["kalman"],
        "extractor__honeycomb_kwargs__use_dead_reckoning": [False],
        "classifier__base_estimator__n_estimators": [300],
        "classifier__base_estimator__max_depth": [30],
        "classifier__base_estimator__class_weight": ["balanced"],
    }
    
    # ===== MiniRocket (Temporal Honeycomb) =====
    rocket_param_space = {
        "extractor__acc_modes": ["raw", "smoothed|velocity|jerk"],
        "extractor__rotation_modes": ["quaternion|angular_velocity"],
        "extractor__tof_modes": ["pooled_stats"],
        "extractor__thm_modes": ["centered_diff"],
        "extractor__sampling_rate": [20, 100],
        "extractor__maxlen": [120, 200],
        "extractor__window_size": [40],
        "extractor__clip_value": [50.0],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": ["kalman"],
        "extractor__use_dead_reckoning": [False],
        "classifier__base_estimator__num_kernels": [1000, 3000],
        "classifier__base_estimator__alpha": [5.0],
        "classifier__base_estimator__feature_selection_percentile": [20],
        "classifier__base_estimator__class_weight": ["balanced"],
    }
    
    # ===== 1D CNN (Temporal Honeycomb) =====
    cnn_param_space = {
        "extractor__acc_modes": ["raw", "smoothed|velocity|jerk"],
        "extractor__rotation_modes": ["quaternion|angular_velocity"],
        "extractor__tof_modes": ["pooled_stats"],
        "extractor__thm_modes": ["centered_diff"],
        "extractor__sampling_rate": [20, 100],
        "extractor__maxlen": [120],
        "extractor__window_size": [40],
        "extractor__clip_value": [50.0],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": ["kalman"],
        "extractor__use_dead_reckoning": [False],
        "classifier__base_estimator__filters": ["32-64"],
        "classifier__base_estimator__kernels": ["3-3"],
        "classifier__base_estimator__pools": ["2"],
        "classifier__base_estimator__dropout": [0.3],
        "classifier__base_estimator__spatial_dropout": [0.1],
        "classifier__base_estimator__l2_reg": [1e-4],
        "classifier__base_estimator__learning_rate": [5e-3],
        "classifier__base_estimator__batch_size": [32],
        "classifier__base_estimator__epochs": [20],
    }

In [5]:
results_list = []
fitted_models = {}

# ============================================================
# BASELINE 1: DUMMY CLASSIFIER (tabular)
# ============================================================
if TRAIN_DUMMY:
    print("\n--- Training Baseline 1: Dummy Classifier ---")
    dummy_pipe = make_baseline_pipeline(
        feature_mode=FEATURE_MODE_TABULAR,
        classifier_name="dummy",
        target_col=target_col,
        feature_kwargs=baseline_kwargs,
        random_state=random_state,
    )
    dummy_pipe.fit(X_train, y_train)
    fitted_models["dummy"] = dummy_pipe
    dummy_eval = evaluate_holdout(y_test, dummy_pipe.predict(X_test), target_col=target_col)
    results_list.append({"Model": "Dummy", "CV Score": np.nan, "Holdout Score": dummy_eval["competition_score"]})

# ============================================================
# BASELINE 2: RANDOM FOREST (tabular + optional augment)
# ============================================================
if TRAIN_RF:
    print("\n--- Training Baseline 2: Random Forest ---")
    rf_pipe = make_baseline_pipeline(
        feature_mode=FEATURE_MODE_TABULAR,
        classifier_name="rf",
        target_col=target_col,
        augment=use_tabular_augment,
        augment_kwargs=augment_kwargs,
        feature_kwargs=feature_kwargs,
        random_state=random_state,
    )

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        rf_search = BayesSearchCV(
            rf_pipe, rf_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose,
        )
    else:
        rf_search = GridSearchCV(
            rf_pipe, rf_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, error_score=error_score_constant, verbose=verbose,
        )

    rf_search.fit(X_train, y_train, groups=groups)
    fitted_models["rf"] = rf_search.best_estimator_
    rf_eval = evaluate_holdout(y_test, rf_search.predict(X_test), target_col=target_col, verbose=True)
    results_list.append({
        "Model": "Random Forest",
        "CV Score": rf_search.best_score_,
        "Holdout Score": rf_eval["competition_score"],
        "Best Params": rf_search.best_params_,
    })
    print(f"RF Best CV Score: {rf_search.best_score_:.4f} | Holdout: {rf_eval['competition_score']:.4f}")

# ============================================================
# BASELINE 3: MINIROCKET (temporal tensors + Ridge regularisation)
# ============================================================
if TRAIN_ROCKET:
    print("\n--- Training Baseline 3: MiniRocket (Ridge) ---")
    rocket_pipe = make_baseline_pipeline(
        feature_mode=FEATURE_MODE_ROCKET,
        classifier_name="ridge_rocket",
        target_col=target_col,
        classifier_kwargs={"num_kernels": 500, "alpha": 1.0},
        random_state=random_state,
    )

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        rocket_search = BayesSearchCV(
            rocket_pipe, rocket_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose,
        )
    else:
        rocket_search = GridSearchCV(
            rocket_pipe, rocket_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, error_score=error_score_constant, verbose=verbose,
        )

    rocket_search.fit(X_train, y_train, groups=groups)
    fitted_models["rocket"] = rocket_search.best_estimator_
    rocket_eval = evaluate_holdout(y_test, rocket_search.predict(X_test), target_col=target_col, verbose=True)
    results_list.append({
        "Model": "MiniRocket",
        "CV Score": rocket_search.best_score_,
        "Holdout Score": rocket_eval["competition_score"],
        "Best Params": rocket_search.best_params_,
    })
    print(f"MiniRocket CV: {rocket_search.best_score_:.4f} | Holdout: {rocket_eval['competition_score']:.4f}")

# ============================================================
# BASELINE 4: TEMPORAL 1D CNN
# ============================================================
if TRAIN_CNN and HAS_TF:
    print("\n--- Training Baseline 4: Temporal 1D CNN ---")
    cnn_pipe = make_baseline_pipeline(
        feature_mode=FEATURE_MODE_CNN,
        classifier_name="cnn",
        target_col=target_col,
        classifier_kwargs={"epochs": 20, "verbose": 1, "filters": "32-64"},
        random_state=random_state,
    )

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        cnn_search = BayesSearchCV(
            cnn_pipe, cnn_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose,
        )
    else:
        cnn_search = GridSearchCV(
            cnn_pipe, cnn_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, error_score=error_score_constant, verbose=verbose,
        )

    cnn_search.fit(X_train, y_train, groups=groups)
    fitted_models["cnn"] = cnn_search.best_estimator_
    cnn_eval = evaluate_holdout(y_test, cnn_search.predict(X_test), target_col=target_col, verbose=True)
    results_list.append({
        "Model": "Temporal CNN",
        "CV Score": cnn_search.best_score_,
        "Holdout Score": cnn_eval["competition_score"],
        "Best Params": cnn_search.best_params_,
    })
    print(f"CNN CV: {cnn_search.best_score_:.4f} | Holdout: {cnn_eval['competition_score']:.4f}")
elif TRAIN_CNN and not HAS_TF:
    print("Skipping CNN — tensorflow not installed")




--- Training Baseline 2: Random Forest ---
Fitting 1 folds for each of 4 candidates, totalling 4 fits


AttributeError: 'dict' object has no attribute 'set_params'

In [ ]:
# ============================================================
# FINAL SUMMARY + BEST MODEL HOLDOUT EVAL
# ============================================================
results_df = pd.DataFrame(results_list).sort_values("Holdout Score", ascending=False, na_position="last")
results_df.to_csv(results_dir / f"baselines_summary_{timestamp}.csv", index=False)

print("\n" + "=" * 50)
print("BASELINES SUMMARY")
print("=" * 50)
print(results_df.to_string(index=False))

# Full report for best holdout model
if len(results_df) > 0:
    best_name = results_df.iloc[0]["Model"].lower().replace(" ", "_")
    key_map = {"dummy": "dummy", "random_forest": "rf", "minirocket": "rocket", "temporal_cnn": "cnn"}
    model_key = key_map.get(best_name, list(fitted_models.keys())[0])
    if model_key in fitted_models:
        print(f"\nDetailed holdout eval for best model: {results_df.iloc[0]['Model']}")
        evaluate_holdout(y_test, fitted_models[model_key].predict(X_test), target_col=target_col)


BASELINES SUMMARY
        Model  CV Score  Holdout Score                                                                                                                                                                                                        Best Params
   MiniRocket  0.633664       0.631999 {'classifier__base_estimator__alpha': 0.1, 'classifier__base_estimator__class_weight': 'balanced', 'classifier__base_estimator__feature_selection_percentile': 50, 'classifier__base_estimator__num_kernels': 500}
Random Forest  0.480134       0.521685                                                             {'classifier__base_estimator__class_weight': 'balanced', 'classifier__base_estimator__max_depth': 10, 'classifier__base_estimator__n_estimators': 300}
        Dummy       NaN       0.000000                                                                                                                                                                                                 